In [1]:
from transcription_pipeline.RateExtraction import AverageAndFit
import pandas as pd
import matplotlib as mpl
mpl.use('TkAgg')

In [2]:
dataset_folder = '/mnt/Data1/Nick/transcription_pipeline/'
key = '001'

embryo_list = {
    '001': [
        'test_data/NSPARC/2025-08-29/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',
        'test_data/NSPARC/2025-08-29/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo02',
        'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-09/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-12/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',
        'test_data/NSPARC/2025-10-23/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',

    ],

    '002': [
        # 'test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo01',
        # 'test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo02',
        'test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo03',
        'test_data/NSPARC/2025-09-12/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-12/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo02',
        'test_data/NSPARC/2025-09-19/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo02',
        'test_data/NSPARC/2025-09-19/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo03',
    ],

    '003': [
        'test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo01',
        # 'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo02', # Low background
        # 'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo03', # Low background
        'test_data/NSPARC/2025-09-09/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-09/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo02',
    ],

    '010': [
        'test_data/NSPARC/2025-10-01/MCP-mSG_His-RFP_RBS(010)(trans-het)_embryo01',
        'test_data/NSPARC/2025-10-01/MCP-mSG_His-RFP_RBS(010)(trans-het)_embryo02',
    ]
}
test_dataset_name = dataset_folder + embryo_list[key][5]
print('Dataset Path: ' + test_dataset_name)

Dataset Path: /mnt/Data1/Nick/transcription_pipeline/test_data/NSPARC/2025-10-23/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01


In [3]:
# Specify here at what frame NC14 starts
nc14_start_frame = 0

# Width of averaging window for time bins
time_bin_width = 2

# Number of bins you want to split the full embryo into
num_bins = 40

# Load in compiled dataframe
dataframe_path = test_dataset_name + '/compiled_dataframe.pkl'
compiled_dataframe = pd.read_pickle(dataframe_path)

In [4]:
compiled_dataframe.head()

,particle,frame,t_s,intensity_from_neighborhood,intensity_std_error_from_neighborhood,x,y,ap,ap90
0,1,"[104, 106, 107, 108, 109, 110, 111, 112, 113, ...","[184.3107988999989, 187.63377190000006, 189.52...","[140.40912258064515, 121.79283216783219, 213.7...","[52.03203548803622, 52.805172929835074, 50.344...","[605.3765813734504, 606.7073925961548, 606.137...","[60.845705956703775, 61.094982945930575, 61.54...","[0.29989282599277306, 0.29935082914950334, 0.2...","[0.0882810203909283, 0.08792474020484398, 0.08..."
1,2,"[460, 467, 468, 470, 471, 472, 473, 474, 475, ...","[803.9901298999991, 816.4254758999999, 817.905...","[103.61443382352942, 156.52117901234567, 170.5...","[58.37107838517772, 46.98102439360665, 46.7775...","[794.38524199895, 793.5522284837637, 793.42755...","[126.61723696807947, 127.56016297232702, 126.8...","[0.22369606202316095, 0.22406356366704563, 0.2...","[0.0051179927154540675, 0.004162635184571489, ..."
2,3,"[423, 425, 426, 427, 428, 429, 430, 431, 432, ...","[739.3221608999986, 742.8008869000003, 744.695...","[520.9520769230769, 470.851303030303, 602.6857...","[52.09609636655261, 51.4140963219018, 53.22461...","[822.024900337528, 822.0673043307174, 821.4483...","[250.47087823939577, 251.41095105413564, 250.6...","[0.2154891483234969, 0.21549583310951625, 0.21...","[-0.1295334218694125, -0.13054425880401702, -0..."
3,4,"[19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 3...","[36.62122289999947, 38.36057589999959, 40.0999...","[228.155, 202.87367567567568, 145.251208955223...","[44.55221627483868, 48.799561465386404, 45.737...","[839.7632086637024, 839.5969738022496, 839.488...","[74.65014406836833, 74.04090304719372, 74.8362...","[0.2036611431160635, 0.2037139887279898, 0.203...","[0.05780576469496439, 0.05847014482216729, 0.0..."
4,5,"[2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, ...","[7.052250899998471, 10.530951899999753, 12.270...","[329.18301503759403, 295.20733333333334, 353.2...","[52.37345586056783, 46.105959021945665, 52.598...","[873.3252270331539, 873.3174784917752, 873.884...","[77.11092641252668, 78.44701566337275, 78.8017...","[0.18989408858241355, 0.18993161743721315, 0.1...","[0.05292289248664111, 0.051490777098614444, 0...."


In [5]:
aafdata = AverageAndFit(compiled_dataframe, nc14_start_frame, time_bin_width, num_bins, test_dataset_name)

Load previous bin fit checking results from "bin_fits_checked.pkl"


In [6]:
aafdata.bin_average_fit_dataframe

,index,time_bin_centers,time_bin_means,time_bin_stddevs,time_bin_stderrs,denoised_time_bin_means,bin_fit_result,bin_fit_slope,bin_fit_result_modified,bin_fit_slope_modified,bin_particle_number,approval_status
0,0,None,None,None,None,None,None,NaN,None,NaN,0,-1
1,1,None,None,None,None,None,None,NaN,None,NaN,0,-1
2,2,None,None,None,None,None,None,NaN,None,NaN,0,-1
3,3,None,None,None,None,None,None,NaN,None,NaN,0,-1
4,4,None,None,None,None,None,None,NaN,None,NaN,0,-1
5,5,"[0.016666666666666666, 0.05, 0.083333333333333...","[188.5157701202671, 179.42234069129577, 265.02...","[78.52885956591007, 25.055866185817266, 55.158...","[18.716406184442786, 37.11461124429882, 30.162...","[214.5059727623684, 216.07088544099724, 218.31...","[[0.016666666666666666, 0.05, 0.08333333333333...",261.817796,None,NaN,4,-1
6,6,"[0.016666666666666666, 0.05, 0.083333333333333...","[278.81242505935927, 261.9223620899314, 270.23...","[153.84097070677606, 138.5476561132398, 138.54...","[5.997866148215882, 11.730082602508134, 10.873...","[271.3831246617401, 271.60245700602167, 272.04...","[[0.21666666666666667, 0.25, 0.283333333333333...",112.076352,None,NaN,56,1
7,7,"[0.016666666666666666, 0.05, 0.083333333333333...","[227.2467448213174, 221.59367156481235, 232.29...","[104.25764455656227, 103.44437117382994, 99.69...","[6.866548598837019, 11.499745606977603, 12.480...","[263.3764150665997, 263.6976685299611, 264.361...","[[0.016666666666666666, 0.05, 0.08333333333333...",111.011998,None,NaN,53,1
8,8,"[0.016666666666666666, 0.05, 0.083333333333333...","[232.9293933090342, 228.76907584005434, 265.72...","[137.08798848334476, 105.23134870622069, 127.6...","[6.550354299840012, 10.63511563804607, 12.1589...","[278.7121983204904, 279.09556443610956, 279.89...","[[0.016666666666666666, 0.05, 0.08333333333333...",128.220535,None,NaN,59,1
9,9,"[0.016666666666666666, 0.05, 0.083333333333333...","[216.43499372687586, 227.28941765345445, 223.5...","[99.29330315770248, 135.99605770985627, 102.28...","[7.121713872219973, 13.035858490085353, 11.221...","[260.9189257171718, 261.41417711758066, 262.43...","[[0.016666666666666666, 0.05, 0.08333333333333...",156.954594,None,NaN,51,1


In [7]:
mpl.use('TkAgg')
aafdata.check_bin_fits()

/mnt/Data1/Nick/transcription_pipeline/transcription_pipeline/RateExtraction.py:1518: UserWarning: No bin has been left unchecked. Move to the first bin with data
  warn('No bin has been left unchecked. Move to the first bin with data')


In [ ]:
aafdata.save_checked_bin_fits()

In [ ]:
aaf_ap, aaf_slope, aaf_slope_err = aafdata.plot_bin_fits()

In [ ]:
aaf_slope

In [ ]:
# Load checked particle fits
fits_path = test_dataset_name + '/bin_fits_checked.pkl'
bin_fits_checked = pd.read_pickle(fits_path)

In [ ]:
bin_fits_checked

In [ ]:
print(bin_fits_checked['bin_fit_result_modified'][6])

In [ ]:
print(bin_fits_checked['bin_fit_slope'])